# baby-ducks-5s: video-director bundle for Google Colab

Generated by the `google-colab-runner` skill from the manifest `baby-ducks-5s` (manifest tier **draft**).
This notebook installs ComfyUI and the custom nodes at pinned commits, downloads the model files,
renders every shot's still and video (plus audio jobs), and collects the outputs to Google Drive or a zip.

**Spend.** Running this uses Google Colab compute units for as long as the runtime is connected,
including setup and idle time. Estimates (unverified placeholders; check the Resources panel):

| tier | needs | jobs | session min | compute units |
|---|---|---|---|---|
| draft | 16 GB | 2 | 22.9 | ~0.69 CU on T4 |

A watchdog cell releases the runtime when the session cap is reached, after it has been idle, or
when the director sends a teardown request. You can always stop it yourself: **Runtime > Disconnect and delete runtime**.

**Hugging Face token.** Some model repos are gated. Add your own token as a Google Colab secret named
`HF_TOKEN` (key icon in the left sidebar, enable notebook access). It is read at run time and never written to disk or output.

**Licences.** Model weights are not part of this bundle; each file's licence note is in `models.json`.
The LTX-2 family weights are under the Lightricks LTX-2 Community License, which is **not** OSI-approved and has use restrictions. Read it before any commercial use.

In [ ]:
#@title 1. Settings
MODE = "package"  #@param ["package", "orchestrate"]
#@markdown `package`: render the whole bundle here. `orchestrate`: start ComfyUI plus a tunnel and let the director drive it.
TIER = "auto"  #@param ["auto", "draft", "standard", "max"]
OUTPUT = "drive"  #@param ["drive", "zip"]
TUNNEL = "cloudflared"  #@param ["none", "cloudflared"]
MAX_SESSION_MINUTES = 44  #@param {type:"integer"}
IDLE_MINUTES = 15  #@param {type:"integer"}
BUNDLE = "baby-ducks-5s-google-colab"
ROOT = "/content"
import os, time
SESSION_T0 = time.time()
os.makedirs(f"{ROOT}/{BUNDLE}", exist_ok=True)
print("settings ok:", MODE, TIER, OUTPUT, TUNNEL, MAX_SESSION_MINUTES, "min cap")

In [ ]:
#@title 2. Unpack the bundle (embedded: workflows, jobs, runner; assets are separate)
import base64, io, zipfile, os, glob
PAYLOAD = "UEsDBBQAAAAIAOIscl1hBz/ZkQsAAJIeAAAPAAAAY29tZnlfY2xpZW50LnB5nVltb9tGEv6uX7HdfgiJylTSHnqBfOrBlzjX3LVJkNgtCsUgVuLKYkyRzC5pxWfov98zs8s3SXbQqqj5NjszO6/PbL79ZlJbM1mk+UTnt6K8q9ZF/sNISvlrmqcblYkXxWZ1d/lanL17LZZZqvNKBO+YTNgqydKFKPLsLoxGo98Lc2PFNq3WQuV37UKrza024vL9L1ORFUuVjYUS/y6K60yDJlMLUdV5rvG6MEKNrM5WJ+vCVjqZLLOiTkSa20rlSy2qtaqE/lIWVls8aEHvE2WSVtbPFxfvSFNog1cbfLXTkRCmzgX/bL3YpBXUI6KTVWE24LiF3qus2I7FVqXVWCTFNs8KlYiirsq6sqN2XXtDWz4VpUnzyuJSbMoqThMQEgeWxDcQAFJNG9sURneUVgRVutEQIIyuapNbkaW3vKGqtmORF6ykNqYwIdiudLVcE9t93USxgjVX8JVd62SgypJslon2RnVf4cG3Hy7EBOpDRF1WIl2RkfI0vx4L9+1zrWstEp3pStNnfk5ImbpkDdobKIpIudYTVSdpMblNE13AaVAPisHJXpijnjBpOBq9RVAYkNqpODmxuhJv3r48j1+/jF6/eXd5Mfvt7JfLcxG4S6mMxe6UFf/58PaN2K51LhAFNl1kGpF3VlfrqSAeL97++uqPGL6N/3v+h6gKvMwTIYmgMOn/VJUW+VT8SyuDiPzHjb77SYqgzqFKukohAQ4bUexCYJUu60wZ4UKRbEcbM6diudbLG5HC9kmxtGEkfoerJ97IViPosNQKiBAqUSUWj1YwvPikMvVpsUZMgu9kU5AuEJol4lOxiCleJnTjGQW/vr4IT8FPwzAXr1+cR5SUo3RTFgaxYa7ZJmPxyRb5WGwQTNVdqRE6Bf63d/hDATYWdY1g8KtqkyFfI7/SPxkNz9qqfeaYG41enr86u/zlIkbaihmYRigPqSny6FpXgXR2xjc5FnJdVeV0Mnn2/d+jp/jv2fT5s+fPJVw8SvRKxBAQLBRJLFW1RnqpSs3eIC2gtoZfEv+w1gr2tf7JqO3slcpomc+U2Y9PQ0pmQZpCJ+IZGVuZtAzkRIbiOxbAFGt8v5eXKD0nZ9coWXIqpL1Js8yeLKlUnLhKNnkmd0yP+D6+RR9K0ktm3vNhPMkrCJM+piS06DjN97hcNcL8XntMo7qEYXTgv4QNZWqb8heQ4cYiSNIlvJWltgp7WtFH6EHhECX1prRMHkKTZZHoIOyr/6JA3ufVyQUixmmvyjJLl7ydCbGQTG7waRgl0Xt3DfDae9Jp5T3pLp0v104u94Q9TngsSp0HpnOwv4aU6EbbstvdokjuoAy9BAOVBK19ECcdmaulTD3qPbNRqPjYgD6FtI45asSXuN/5SCWKuGkGgQtWFBU7CxpDb1eNiYk2YP2JLmzVkTmsbSXKH4g7xZCQkf6SIqwuD9rOKbc0pCcaBhvqyQeFVhAgaMInqKAIcen4UzuxxJq06pijjI1FPBa3itLCRly8yJWBnMnO8XmaMBkqM8iwqE8Y9QixDdByD9rbxmArK8l1eypoy+IeS3btGr+1HtPK3A35OG17rsGLjlx/WWq0pd9UVutzqkjDxaWytn2xXc0h/CqCMnCjqjOYmduPRXW634VzPFCUQ0A/KrYr73fX032N2q7GHuagT3It8r6nZOjVMjlx3ZRESH875dWyXY4X7T1hAKrFEf35WxBGa/1l10WxqzlkyZgLsO1XHKNSxOn7OqcEYWsEvTw388G6Kx+Mfpf42rZ91B+/Y4YZTVVOEy9rPdziSk7WqDKFuZvcg2gnW3XxRG5e9wK8wtr1vCQ/8FacCOeA/fy8l2ybkgKyIUSB5g1I4o/y0GMSo8ajvM9mLQnnrUzgHDkexAX9pIdGYNlXqHnLGkHuRlsLJEJkjbj21VjMr8L5yQ/TK9cePu/7nsFQZw9AhiCt9Gb+7IrUJPNQrtIrstNnx54XxR5mORnhQe16wDbNqt1fEYkylfx5kQ7weYmPk9b5TQ5gKptKSmCmiy7bVvjYook/xcsiy3D/fdR0dECb3xvArLJMYFEk3ubNOqEV4C+22IBlQl89wHzqkLIDXUZTNdUcoWWmljoi4ERSEnSOLAUenzHjiP4EhBta9fxeLcEHt/HtmiDshal1ZzfS0qcAbW9Ymrr0ID5Ae3Q9TIyQKAIfw01gh9ODaF6iV6d5rQcfDoqpV5vFUA/Yz+8Bsa+t53xB9ad2q6dCfMuzygKIHvpvaqQ0lXPgjxsethYKM8hjUh8IDp5dXHXqdkpJZwId7vpdB54PzNetNB7yDNkfxls8uqWWYYNwz5Y+gAk2BQOyvgL9qPhp1sbLkNNRaUfcMehHxDSJEWPQfn+LvmkeBEP4JzfA2ttM6zJwCdZAcLZWFw1jGh7jJDVea67dx0MGUHajbjRobeAXjRE/0CEubmaUFR6ZIEkoaeZXLVDJvSDGK9WRGhxRrRqYjpbdpHnirMGDItHK63TFV54s+Y4nTQ4EEryfN8RnRUwgjwUTUy5+1IXnV4fOIshDXujANho5AW1ewVJytdGy8dXqkAX9jiYr/T53SJqnL0K/Hpnfd9ynYjXvnq4oi+rFqsgApembD5v2Fb7L8LD/PfaTNCR2vPhp3HRMGe7CA24JR0dBqLJaR5+KNO8CYaDt4VIGs4yRE5ov5XYhGdqv1sett1pHW4OgCAat9jbV23/SWPXZDYUUdP2Z8Gl4KJmjIsJQg8ZHwgeIiD/6zKj0l4ojm8Ky60UXeE3pjdSFujS8M6wHUkQbwg7eGU1qneV3IqhTwtKwLNbAHUJH15E4+/AeIP16o+l8iM4s2h5EGXEkT3y0khrH8+ILU3i3kagH45mC3m/9npEh/M1C3LqpaHCntwfovTXc0cMBLHykPjBWmzeljJFIC1U6rQb+bM+cesg5bhrHrp0kJR0aeYSls2NyPD55QIxDaCTCHWGBP/ep4xKOwZutctiwE0vYnk2UkdyhRlyyOmzXaBc2kMidfg0OQopbbTjgXSF1G1lQpdibE/jDkpIVH9ujnuga9d/GdO+G0PnTK65W/WG+WFa6OkGv1WrjApCytZfRpBC9ciycCm7Y5gHvfrH7aD7mzYnBy9SWhU3dMRrNsSc0/p8y09lHV7I/ylPRlAW8u6fr7qMkNnJYpgcHEVNxz1tkeUzcHV7QoUoza8O2BoXEHwPgy8c+UyhNa/+s4q0rnJ7HdL1/UsFLT/ioqKF2g8iTFZ1TPdm1gk9O9vTvx9ggSvtnoogasvxefB7W9+Zs5X54joNtyQ0KVkpT/aS3xUVR0yH53QyayV2DCTYKlVyZ69v+jEsHA83RYnRmrmsqYO/oyQSof0uTMmCcxXFSLON4LNxRBhI6XmYYyWft4vdq+7Jb8LPOylcNqbNGGakkiZWXESDU0BJhAo+YZr3zR7fAknJuFVogSwEogVbVTC43CeEq/blOjU76uAT5sGxzU7qOimG/jxgIt1jH2HENlnTwuqdfd6Rx+I3PQcBcLdk60lXf3mbmve54uNihwoaWmvFxIb7ngZbyZAZtVNWte/YcrXB0fEOSBrJjTDG9gF2OF3Ymv/srYn98RCrjzuNij4s6ZomHmLtK/BD3h1a5hDu2ikqWX6c40ngJEVjOk27YjxBuX4kpmgBnzcGSivi0dHi8qKLun50UjQrDScQJoTbnme+df9G/PPUPgQ47aYg9upLTzfvQiUdzrxE3RBLfHLyig/SVMHsdl+eTvbFq7iE4DYBuzvDMuYmqiNjur2DUxSsc/jJHpoPeoVFUFmX/KxesztT7pjB0xJnAp7NnYQ8/dBblZJg+zGB+fzMVt24aGeOGRr0GlJFZbsQ3s+6UadcNhH3bqogPPjrbXn1NLZctj+g1MC/xb+z7Nc4+VR5h7ZFfj3d4nJXPn0dYeaDTsqLwIGajERjFMTXcOGZecUxNKI49N+5I4ej/UEsDBBQAAAAIAK0tcl1Q6qdRuwQAAMELAAAJAAAAam9icy5qc29ujVZbb9s2FH7PrzjwyxLAYizZTt3kqWnatFi2bEU9FBgCgZIomzFFaiRlxwvy33dIXXyLsz4EMc/1O3c9n0Bv2buEsI8/Cip5zoyNeYakXkKTdZBV6cIEY9PbEbCcaSeSaZpbz3IUg6TnE2jJ9QO1uIyXmhbxLHGeLvqe+qgSJ/+3e9SCSKwdm0EYG8uF6PUb+oLLmrNDNXNlG/mOplVlWSdKclE9RfEo6fgrpRe5UCsn0v425x7weeeX0JKTR6Nkp5ZzwTzcB0946R+BveQZUwewd6lvwvaihEfLn0ZcaxxHXL9brN6X8K54QWeslUe6pXrG7LYOEofvx6QWbGkPG41cq8JZwlpeborWsF/q/03CTjrNXqEyJjZueq5GQRQsBOMyGCVBXk6IoTmzTBqlTYPQi0XBkrLXmDNWFHQUhFESrLidB6VWj4GwaJeMgyQPL15TavkRamXcg2dZML0J/hzFv8Y/7shsVuX7wrTKuHIw/tesoMiwgSmp5VQEVWlSKpgOMAqnGoRk8Ja6r6t3lCq5POrtnxWT8RBbPMa8xQV/YtmO2FbipbJNE9dvnGQsre+75yaNZeXq+X3UWj8YdN9ATGY47XFKS+S963e66F5iH5s4Gs2SOBXUOHdDMmxEXPu2MsiYkKhhGGarcosThqQ1m6mVFIpm9fYYvSPjTscYruSWVhSR9w0zreKS6XiuKgc/JJOWrooSpeNKct/pA3LRqiDYqigtmnSMnsbEmD7c/jEFUzKWAZUZfJyeO5NANYNKLpnmOUdWKWjK5kq4rFxBOmfpAuycwa1SM8HgoxI0gW/MoGrKDJRUsnpM3Fjgn1snPn87W7Tdy3XhHKYde4jMQEHLEgFYBXW7ZFyz1CoN3hKBrTzAkooKvTvsqKXVky8+lFUieAo5n1XasTFMJzL9/a9P375+/vrp5nI3jnRO5QwFMcDCS+M+W/knCL5k4BIHXAImeD9mAveYM41IwU0pBEFaBYgv8PgQdXtgoN5FJHUeyVYQpCkGR6BQVMLyUmCkoKQH0Ib/CzpEs3WjgmE4QZmB09Wcp3PwhcYcQDSC2+vAt6nL5tkVUGEUxrRd201bEN//rk5tifykdMOD2Uznm+3peCdbC7N3cAOPjNex3vWRuzbvRqDti7wMLzAFYn0FUoHbFOduicKNM+tbAagFU1AhwPB/GTbp3fcfOC/XINEmRn87/exrWdA1+EUI9/e/kbpH62Vx90akd0cjjUZ7kRqLbqjOXg92dBBs2O2INlhfNYwT554KOKUJnk43+mO45ddQGZoIdnYJrSMf/hVkWpVuTHymgefYnqAracBpqxwKVii93on4QzgYHI/Zc49EPRrsRV3QpyPVPSzv4KC8o4GL+NSoguHCxmGaOMKZw+/fO6i/vIn6yxuoJz+PekIGB6iHe6g/LCnH6eWC2zXuHs0ReLJ2u1KSzfLzI1XJhcQ1303VBlPt5rU52YcUNZAOxqQDNK29uFG/hFLzFHefuyGFO/14mkVd/hpSe10ULoI5w/OzOTM1xkQpa6z2J7D5pHVw8O61F8tnJYwatAo/LlIbY1PiZ5IqY9y1TC8dsOHWEcCUx68ctrHncaTLWYwbqflMezh5+Q9QSwMEFAAAAAgArS1yXQmyVkODBQAAkw0AAA0AAABtYW5pZmVzdC5qc29uxVZLj9s2EL7vrxjo0hawHdvJBs3mlKRBLjkEbW5BIFASJTFLkSofuysE+e/9+JBkr5M+TgV8sL4hh/P8Zr5eUWHrng+svOPGCq2KGyoOu32xgaQ2nDneBOi4Pz7fHg7bw68fD8eb/R6/eEREacWqadv4+tZur23EKyN4C9HXK6JiNHoYXTh4TZbXWjXkuHV0JxquN4SzXe/oVjR2q7yx3Ez09DeqmXFaK7JukvyGGNlBSKE6Cq9ReC1+2VEy24d/QuHQ6JtG8mADFbdCRfOs02ZKkOSqc31pAV9HgNmR19G2w/ObF+lQOwb58Vn8iM9HJ5OZdS9kY7j6yVI2dmv6aeAnJm/I6taR0V41vCHbs5Fb+Ck66qS2diI+BcAy502IMI3MOi6p1lIbm683bJLhxQ3VkjPEQQyj5PC+vu2i6mQs843QOdJUKGagMeWxZdLyTYQHb0V9Dtn24RyQYiztpB4dk0x1nnUxAtwWwL7FZ2s2hmdCoArdtsmWhkuBOmKV5EHwKWqIccW/z+meHgZuasFk6S0/fQu3W+als0HQLNeRDKSiSNakbNGSKjisKUaAfg4l9UtGNXIlFENEe2ZY7bjZkNIO9VHrcSLdElPIwoOwLlaUYQhnsPEqulc4Zjru5vqd66jTupO8RJZYlRz2RgZB79xob548+f3th/ev3rzdOTPVUvumlczwHVxOp7vRh9PvkzfFnWFD2VVrqVW+WV9F0thDiSvlIJR3MaDH680iQXM2wgX0sE9oQkqvRKznd9FaehOshdvDCCUUhDYHqeGsQQvxkwde7EOC5yAIboKixrDWFTNSghRs4glmrR9QvYfn9O71S2LNF4+uxmFuaDS8jeWbmITFlvqaGkpICS0OFBHN36zoHTOCKbfCkSJODu9zESnkzQs3QdRxtUpCZNCcDcgM2LMFO4/jIfU+us6WA3oL0NPd/gQTATrmOHQs3YrWV6JKdMC80ymPICB1jqBKU8WwEdx3x3OrtqEkl5NZu3XorkW9UI7dRv2NVpnG1lACDppRskmyGHNyOBuzIjGy9uJujOwFukb2QjQTzTk6antpF2LIh0pe6qiNcOJPz08FOQ6zL2liMNdfDJYn8chuyNGUKG9Ubzj2ikCx48VQ4PZ0JuyyCSwaHMkl9tk8xObLqTtCe1iYO2J2BOkHMW6gKhLvowl0L1xPrfRtO9HEpdT31GJw9tyck34lQd2Z+hFk0BBopwOjc3a7oXuO2oe2MOeYlIRmJsNgf8/cbBIKoRRDYmPlpcyw5ZEvAxKAbwvZ/s3sWqdVjqZ3J0qLOy1qvuhMhdprN7N6IqgUN7s/ZEKBI7EWUvQziPo2Lg7c/S4TVePTjFrHMKhlGlNb1XF6JbAFQ4aMAAfPCD9kvGaYISyNdmiqMTFjgyDTtt8KtRxbU32S3/D5OU83Xc/TsrBeIScdMw1XFAowJfZspTgfsEsK8jBdvzF2hCpb7D1nsEPCrYiuR4Ypau+WMKFLZ+pHpjWoKjkIfIfqejiWz6q5EILb/qKnz1at1/96XaH0Bv2vJY4UOgzh3LGV9DzHnQbRbFNFJTPuWZgvjdHYhlw8DzuICYz4L3pqvSSNoGzDk3Fl5Dv6I6XWcGT2JMF50crLVwuvoj8/WLdO97UfLGqpSCl0yq74QXOmdM/s+zjdEd+J491/yPRHuL9kqoKpNYgPsVxZMAQoRXPzOHytnDbUYxPAzcpGHhVyekmYqg7un/fVPzh1aWvaJkKzrXtqJJOTCZGXgu+5WTy0rOErdXx/6+QPozaJmhLtIZjjQmeZveo0p+Jj64KQtGJxBKs6/mhLWMgQnOSxr55L09VQDra05RgXo6B2dnGWYe2MmfUKm7FoBaon7k3ZkWwehlm0/+rbX1BLAwQUAAAACADiLHJdJ6t4+RkLAAD8GQAACQAAAHJ1bm5lci5weY1YaY/jNhL97l/BKBi0hNhqe5LJDtyrBDkmB5ALvcmHwDAEWqJstWVJIWm7Dcf/fV8VKV/t3k03YF1kVbHq1asiP/7ofm30/ays71W9Ee3OLpr6014QBF+v67xSQq/rWmkR/sZfhLF5Vc5EU1e7qC/mCt+kVbmY7cS8aeaVGmRNJWcDNy3u9R7XtRF2ocSM5d0Z8dTMjJBzWdbGCim+aVbF7o8fhVF6o/SYJOWqkOvK8rSmVnytG6tmTbOEBVKTRhiD973vWSukQCsZa8uVEuHC2nZ8fz96+694iP/R+P3o/fsoFj9aCLcqs86m73/7oy/aMlu6x0U5XygY1cpsKefQYUule3YhrShKa/pi3VaNzI0o63ZNz2Y9W/GHtqkqXPJmW7sRzdrSECHrXGiVNRrvFNYHL5b1HAJ6fsQ91mNV/GSaGvIaOEQr8p6QhYXfpch107awxShjSorAsmwN7KlLs8Br8ibcLHzofLzidicGg6rEYq7/PhZm0Wx5abDYBQNGeneI0opts65ydsstsZPBYK0r8cfjT1O6JzlCrm3zd65h8t9YTp1Lnf+9ks88AOvsvMHPeD9YlfXaKiN+mRLSeuWqbTSwoOet1EbBKvZGQx5erG1Zsadb3WTwAe53+KE493o/fHj8IBKMjFtpF3Fe6lquVNg9y5mha5imRVmpNI2iHia7b4Cf0jYc9gUJiTojMoJjmlWlqmGREVlGLqubv+RYfPhs+LbX+/Xx2w+PUDoJeMVBXwTdouke6wumvV4PIPZYS+ftOozGPfK+1Tt3Q3/kmuRsaTG8HE6CelPmpRyYVUnyBoO/1sDNAEISWlx/pVaN3sW2sbJyA4pGr6RNMrPp181CyVxp3KxrQDOY9nsvIHD+l8nWrrVKXYiS3/Ua/rfq+XgLP+Nb8nYYxUh+3OKiyzbEY1uVtiprZcJoMpwe9bCVYgWagJOeu+ECVopnIF+wDJobBv0gOs3TCpbUfrqG/XlYIJtsCFGRuBej4dvPIh6tnjPVWvGBL0iK8bWMX0Abff71kSA0p4TVsK0ksAVvAkRO1UbLVV9sZW19kMqCn8RHiQgI2sEL+fSZ39GiKJ1Z4CSg32Aab2S1Jq+c5kEkKeNUk/UuZEfQm3iN7Nbn7plPgCGbLYLp2Xz6IzvhUr6Q2klAt+l8FjgfEkeRyy3LsiTLYRXK+YmWPgk484Mpm3LxZmKnUF3W6VGs+Hciwk7f0IfKu4C0TQajKUlnzaoyyikkMDi3V9LYtNCUkwC1avoU/LSt535pL7BfFKtWzRnWO/6tiN43ipGutG40vzVGNQXfgd7f8Q0li9cRDFilGW/oyyg4ap32RbZQ2ZKxHZ2vxg/wZq9QnLrotXBpx0zxV3q+XoEZfqMnHebKZAA3QTBJ07zJ0rQvXDaCvMEi0pjkOPlRbr89TfhBVe133VBnSxvLPE+l1xEGzLSw3tfDBLSGIl3qpo7nCt+/+fXn7/5MwcS0zJslL4hek0wxPxPtcE7uaUqEI5m4F1PxiQvpa2Lgt0sDmVufGviPaBV2ee5/3ZKzggBRdteqhNO+LxZwURIsQKxIuKoaZFWTLYmxHMAXpTm2J96CsauHVD8QxOfUF83Uy4+C193BNHdlQLeu0fvhsDMH2TqAko4YKa8MCnydm1eFUx2GZJkxUFAtGhCuBQRfnQEeGRA7BNHDja+UkQPk57mxTpIEWNuY4UYzQEFOAdIcX8gtMfUnYdOqOrwVraP7EC8398SRkBDKGKal7p2MPVFExAGnL44JzmufK33UKSQvmNhREM31NHRJyzKm104CNyvJNWnhCt6ib54HW13CT0WAdmYs9mwR4HJXUyvprFL53UGE+47Z7r7E4/dfR2LwhbNyT780ooKbSHJ0YOUPAFVdFtQjunFsyl33kldwNz10MGOvUPBPPE7AfSLMkLjx0VQh9k+TuzK/m45Hnx/4YVnW9OifUA6tgmgvuRNVv6D1zh/UL+OxUwEN9GYMzNTRdTUb8gvAYSWXCj2UQZihr486C+PT5pwvTXvWbzFy/Njg1Mh6I/nFS9iZNmLAdEJKQ80ZvXbA2fNSgjG72EMST/sD7pHudD+ZHjoHsxK3erAVKjbdnfzdOfhkHKjEIIw5anEXRSfjzkfvQSyVaqlL586fh6AxpwaZ+nIM3qEtbGoD3yAhKUU9tC9sefincPWzeGnTWKIdQOOzD/w+h/2ANgFdVEE3YfDmz8Gb1eBN/vubH8Zvfh6/+U+AnRglEIa6zAmobIyBPVwPLhZEmqhjMZFi130DP2SbqyU32JJXNhSJM4CVRz2WRjXSyI0673DsijQAHp+IIMZD8OACn69XbchrRBmm+OMbTNyS1YA4PJiMoAho0ArmZMoNABxOusD7sgq1Ks70YePwqExTbRQ5enxPrQaX/fE9qq5CKcB4pVWNciYsbaxYiiCwxbTrOGvNMDJmf5ttie1C4AQEV/1X1yrcoE2eQOULkiafj6fR/5JO5gYRuf/6y2kN17qfyhze5fGuc6ZMHkWTEXDWarU5oc8hi4OKSaxmf7iQBZNoios7zULPGnGzm4MiA5oBrjgbQy4z1xaxR2SJhP0J2/J1+4GaM2RarRT2unvoRsJuF2W2EKjTJJClRxdCilfIhHRPvN4p2snoViDOmssCbRcDj97FaOSC6Jbnz/3ryKa45sIb0SXcOSwWEhZRIIbH5v9E5kdJT+ZFNIAO30uET5OgzKnK7Tvfj0VAOQ/GoT6OmR7vMM7dTg8XaHrCNsNPnIqkC9plbI6styzbY2VBPaOhEaU16MsiydW5YKII2hOEZ+kuBmAA2nt9PhRfJDTktiJ8SBclcm6PuwP2fjV3bMJ1bFpJtN35A5zStMysMwXfqZNpZJP3mBdFixNUdh7EDPOXR7Xb4h91MnDfttFL9EZbbKSikwsvduBdFIuSw3iE3Jg2/d1pD7E+n90wEfpTmkIrszgeyiBNjKBNkDsZEjOZLV9kC5FzCuOzLHaiQ2bovqe3ooR63azIXJd3rgN8Iee4u6MZQDcph8k39/l1mfdFSkxL7Gxjjd7QltSHhkFMXt8WE4xBRXKWI9twQ75na48y/Y77LNXpcERdkZTBwnJEMXQATYIZxV/RuQjv3RJUsVBF1NR64PgBJyQAQsrBgevLDah2Uyks5+A+b5MiaCyqtVm4zgV1bHRVx16FQ8tMixi5w70uRtviioPcoK08DZlg6tR1rLw1uDgUoT9ipLPk/eh28rJ0JtZHd5zZMaubKPZ6cudunbs0g+UO+10j58rcucriXrLX76LDFYryV2jXc9MVR1NK0LlCN0NjH0SnagWtldbpUogQCY8UymaLziUtwS+Ppv8HJuwFYv1m1dq0zBOex4oT/u13e6zEnQtdUdSIauFrB10GBSmp1bMNw82pad4Wx1MafrnEhoPPZFzq+WwASx+iuLRqReMQvyWNQdVUDGp02YhTyk8Rk2dp6FhboukIN5R1NorcOdQZA12fXt3IJF9oPknE6OGGv9znq6yaDN4Nh2g9Tsn13Vc//vQisy70vEA//SGIqVZ/dTEM7s0One4qJe28RfdHgu8uZb1+KHeMhE9pvyXpTv7RHqxrrhFyVikRdqTqT76/jGLx6M7EfQ9OO/5Mobx0Z+TgYqryD5fH4kKixCxhDBOdr+/vepfW9K5uCYp8KLsKRwwMRsVlNT8CB4DY3KrG0cU+lEXu6fdwf72jBMBcMA8+5rHThWCZtotWt00jfc6+RBzluD7ms16vh68pb8DTlE1JUzrHSlPPL3TyjQ2dDd3pVtT7L1BLAwQUAAAACACtLXJdVMDVB7UAAAD5AAAADwAAAG5vZGVzLmxvY2suanNvbm2NvQ6CMBhFd56i6axCS1taVhOJD+BkHFr6VYjyE2kHQnh3iwmb27259+QsCcL10Lk5tLhES4IQ/sA4xIwb78epTNNn65tgTvGV/p66H/q5G8KUnrd6u+LDhsWta/0GGq5ZTgkvLAgjpWROEpKx2mZUUaGtY5arAhhO0BpRPOr6Ne32f9bWz0rstmNV3S6bRoCmghMorBPCFlxyJ5wDkBkzNANFncmlkXrXmNDbN9iI3h/J+gVQSwMEFAAAAAgArS1yXTNIEWQQBAAAdhMAAAsAAABtb2RlbHMuanNvbuWYbW/bNhDH3+dTCHnVAaYlUdRT9mrrmq6o12xYMgQoBoGiTjZn6mES5SQo+t13kmw3sVNbdoJlxQADNh+OvPvfj0dZn06M08XpmWGP8EcqFdTY+HRiYEM1t4SSuQKZExaTtAzGNU9BQ14X1WqacZrICn/jV5o2tSzyKCsSUPXpqBut51KpdvLHttmv6iwXrXU31g782c9WBU+gujf96sOby0nXOW5y0FHOM3hgIAW6A60DvORiBoSOLeNVWhWZoWdg/NxMpzKfGudcgNE5ZgheJd8boshTWWVGDGlRATazDCohuTKaGr5bOl9BWbRLx4qLOWkn1pooHtfm+eTqevxQnKVNyfWstdkn32qHhWxFay3cmDErADd0Qp8z7vhCxMxDE0a9JIAk8Bw/ZMxemsZ3uksWs3zLo8ylVt8/5RoS7E+5qqHvairVbjDTuqzPTHPWi5KiJmNRmIOiM3G0UAswh3pp7hMAPfs8WoFGyYLDLrxweDhRo36kAiWnM42h7mTsjx/eLBHDTV6IsNdFlt6Ri2pqnveyve9l+3GDqrpUUkfdOTXRW/Nx7R5jy+Gx4K6wEmGDSBjw1HEwkV6QWKHlOamwRei5G2w5jkdtm4b0SLS+GtY9nIY5Zg6LfA2V0rjd2CWUxiSRHReQkKufyG8seh9dT8bTaZMeWcNwbeoSmfEpEF2QhUyg2F3FsHT1iL19e3W+t5JNWmorKea1Mbm8JtRAGbMml/rOmPSzjFcfLi6Ni9/fEV6WVbEA5A2pMtojjIYaQ6hH2OKJIfUwApu8VoWemd2OKFzr6QZ8ax3NYepuY8iFI1xmCc+ibmg5fsCQMAjDmGL2/dAJXMGEE/sbGNrMsizqsCA4tsY9Ft2awaFemYcKsEVjh0oLLcHqsCBxanvHFb2DERxc5F4Evi+brjK0gV574oeKuI0dC2MrdLgTCju0vZAHPLB9aicC026D5UOI16dD0y3sXEq9MKQb5U9XzSDotqNaIzfUI/OQwLdwU+hwrkmNKqLwpClrwRVUBAtna0zssbULv948WprtrYlEClVUnDR4GtZ38KOgru/nWt8p2MntpPPhqnfhl9aDJcadN98eyI9qah6bsGdDPQxd3w989wVJf15lts4CbxLZH57jqy6kKQhNKhCyBNKCcQ/zg/EvcuzjcyCL9HbzSCyb9azQRFc8r2VH4/+lxO9I1rMh73gs8Dy89/8bxX1HyGuWp5BlnBEbHzpupJ4RTM1fZLXAPqw13OoIclF01Py7Vfz15N2vSzaFkuW3B+cD8czD8vBsxFKPevgwGtjBC5bppyixBvnvG8gjJ2JxlJZBlMlbSJ6K7pNfAxzAaJPP8+ImP+sha98CfPnjvwIOKTu9n5O8UaqN/wQ/n/8BUEsDBBQAAAAIAK0tcl0N5ypjHQUAAAIVAAAiAAAAd29ya2Zsb3dzL2RyYWZ0L3MwMV92aWRlby5hcGkuanNvbt1Y32/bNhB+z19B+NkyLEW27O0pSNIgaDp0yw8UGAaBliiJKyUSIuUkLfK/7yjKnhSRQtP2YdiLbZEn3t13d98d/fUEzU63q9kv6OsJQrOEYSlj9SwIrMxuOE6vS5yT2Vxv0ko0Snai8NjugJhUuFYLUeUzWH85QS9zfegmtB96XxGlDyb11dX9O+vJDYjEFS5beaaevGCx8oJg56VUKsoYSb37C+/3MH4ff7pZ5HmTDTU73Hk4uzSKrUr3mIx07mlKuAc7XsKrvbfL/PVC4owoUkley6HW9c/SipuUGq3TCiO7wvOb648TGhNGxVFlTsoSh54P2D5SVXii5n97BztG2uftAQc9ILbvllKyp0m7mJIMN0wNDI18R3phOFjdC5lgRj7wlLAJs0u9P4KKtUd4UmBFMfMac1btPQWt9Z6/WDrxWy0dZt19evhYE4AiIVJOpv6f+skUUAsEWurPv+adUB4nvBQ1nEJ5BeL+ZqDfkTCXpVDP2giDz4POQasVjzRVBTyt/MBoLAjNCwUrwWZjVhip8lbGD3yzssMqKWJJv2hd/sAeRz5pU67L/I63llxXguHEzgiQsj1QNnZQhshBEMZCJqp9qbVFSqr66N1i2Xn3LMB0WMkwk6Tv3dqBtvbOIN4qPdO1Z3Wurcr4lYs2u7IaklTGVVPuIJd70LcbcQ16dIjCo83OiEQTETnnVYLVWZck9nDoeMVjMCOL0ca7kex66OAAUAe/a/q5I0/qskqgZK2GKdjWoncFQWmTfGa0ytGONxUUHMJViiTkmCzgQYGEaNKUkTl6BONqlNZcMKIkytjzHBUEp/DmTqICC0HZ868oBw8YQZLpWoCXZeHRatERlSa/fvwip3vR6Q+7JxKUQ8TnCJqH5GCTeWrDctiBzsl5BT8KyqC7FXPU5NqxHWvqGr51x+O1Imnnf4nrz3OkNcyRbHaKgq9yjgSppT5GEI0O4NKUGB7bj0fefrUaYAfwhRcYyeFzx1PQkR3q+S34rF0zg8nNlCrgPIirFSHBJezv+6UE6TTOyork+JUcxMVRcv3KGvTJjdvSiwaz83dXVw2d7Dt9SGx22vyxsZ/Fn6Oc35MzlZtkeZ/bTIkeFwfUHdhdfH+LS8iH+pYwkthJQhqJY2cljX7CuhRVjdmgY24dfPQBV4DjLc1LbO+X0myBqPYGLRfb7Wm0an9sDt9+YH50z8uDRGTWw8DvRJcDm063dpv+gDzn5W+cSnuRVnonloSkr1k3dBBbh+V5AxVZnqV7jVHqPrsXY7BxnAu5ybleZm1sTc4o7dN3YBM7AHycSGwkb+g9ft2Do4kqn+hAt0RgXXKTPQjvx00lDJ36wolaPY6LHSTubO4DsbYB0c2J8evihkl1LDw51QyMd6Tif2CACq259aYBKpwYl79/GAltpeEaRqIBVQ6YyXXF+P9wfOhw8WdzvGNOfgPHLxebA3MvO+reWojbUepvIu7V0h9w99px6A9wt61yRty9tRHHmLtDq9iIu223Cgd3h0s3d0/Q0Xdz93rj1Bc5+ubD2eUF0cPyHWUO0C20beOFb6BHBToOd6njvZjvSc2wHmzXYSdGSsEh9Q+io/V/X/HXfSdXEw2qvTsevX2ro31u+Ia75gB5BzWf1wTi5/4DoU2mvjGRjfhaY/odx1YRmZDj0Tty3BFu8X7CqpYY+0bZOlwGcdZ0FouaZPRpZv4TCrqUmGUcbkrtJawU4eFuA0FJ9FIRwGWjM/Pk5R9QSwMEFAAAAAgArS1yXQn4PdMGAwAAUwgAACIAAAB3b3JrZmxvd3MvZHJhZnQvczAxX3N0aWxsLmFwaS5qc29ufVVNc9owEL3nV+xw6QUYAgTa3tKEZDLNpJmS9tBOxyOsta0iS64+ADeT/96VIdTJWLlgtHra9/ZLejyB3nzU+wiPJwC9VDJrE1dXSJbet7vFw61mHE2vH3aFqryzByz0vEKXKFY22Ez63WA8WEsUajBdDbLq/dCyDB0qq41tHEBviyIvXMKfGThmzEvXo82nE3jqBzGn3WIubm/u3xCTSlEdxfzZokomyXSVkIykFDvkHWKePQfp44ON40akUWnjbmnfzxdvKNswfJGl8YAsL+S0Kabx6B9w5xYq1Rw7eRxtB+gnE3IMaSEkN6jeWVDeWDT1wBR1iTC5hJQZp7UC64SUQ7gXVR8YGO0VhxVb1cB9upZC5bAVrgBSnWU11Cil3kKGzBVobB9WIodcamtrWEmWrgFrJLMTqgZtmMoRVsjWfdgiM8EbA1syKcEgB8OEgoK5PqlgilugJSM/HqHynEuEUvCBrSgNxV7Gljk0wI2uJLoGTzqACdOH37rOvARdoRoESuIREoew9Iq05AZRQc4Mp09F6olTZ47+WoeSwtPbJp5UIqOkiJIIKA/pOm9SQmjmvCF2DqmWOkCb85zVMuS6DyVy4UuwhXbDQyeFhqRy/AyLpqsbM4zC7692xWeRimvFhRNaUeJ+oNFfvOtu/BauxTeN8c0mEb6r62svYk1cUtfJlvvRC/f7iCttScgGoyr2MIU5ewWbdcDSLIRz2lYeuRk+L1momFmixLQ7SXaPOM4helq1524WGe2rMLDLtEDenOj07bAKq+nhihPcFUH5aHywFM2lR6az+axNGSn8oqxc3fDeUsspd1OyvHvi36baW1bMpUVixV98lc15pA++0jDq8k4L202qwk5iEXnDPJq3Q4rcXocCXXjrdHnON0yldDzqvdUZk47OyPdtegTNukCHirdQp10okZfMtkDjDpBsypCIpg7/obPohJ1Fn4lLjF7fe8FtLV3DQ09HKzvjmIQPsTpsMN5OTYBtAWcdAjK6V8MUJZXBTOyOj9r6eZpOnv4BUEsBAhQDFAAAAAgA4ixyXWEHP9mRCwAAkh4AAA8AAAAAAAAAAAAAAKSBAAAAAGNvbWZ5X2NsaWVudC5weVBLAQIUAxQAAAAIAK0tcl1Q6qdRuwQAAMELAAAJAAAAAAAAAAAAAACkgb4LAABqb2JzLmpzb25QSwECFAMUAAAACACtLXJdCbJWQ4MFAACTDQAADQAAAAAAAAAAAAAApIGgEAAAbWFuaWZlc3QuanNvblBLAQIUAxQAAAAIAOIscl0nq3j5GQsAAPwZAAAJAAAAAAAAAAAAAACkgU4WAABydW5uZXIucHlQSwECFAMUAAAACACtLXJdVMDVB7UAAAD5AAAADwAAAAAAAAAAAAAApIGOIQAAbm9kZXMubG9jay5qc29uUEsBAhQDFAAAAAgArS1yXTNIEWQQBAAAdhMAAAsAAAAAAAAAAAAAAKSBcCIAAG1vZGVscy5qc29uUEsBAhQDFAAAAAgArS1yXQ3nKmMdBQAAAhUAACIAAAAAAAAAAAAAAKSBqSYAAHdvcmtmbG93cy9kcmFmdC9zMDFfdmlkZW8uYXBpLmpzb25QSwECFAMUAAAACACtLXJdCfg90wYDAABTCAAAIgAAAAAAAAAAAAAApIEGLAAAd29ya2Zsb3dzL2RyYWZ0L3MwMV9zdGlsbC5hcGkuanNvblBLBQYAAAAACAAIAPwBAABMLwAAAAA="
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(f"{ROOT}/{BUNDLE}")
# Assets (reference images, music) travel in baby-ducks-5s-google-colab-assets.zip: upload it to /content or to MyDrive.
for cand in [f"{ROOT}/baby-ducks-5s-google-colab-assets.zip", "/content/drive/MyDrive/baby-ducks-5s-google-colab-assets.zip"]:
    if os.path.isfile(cand):
        zipfile.ZipFile(cand).extractall(f"{ROOT}/{BUNDLE}"); print("assets from", cand)
import json
JOBS = json.load(open(f"{ROOT}/{BUNDLE}/jobs.json"))
if JOBS["missing_assets"]: print("WARNING: assets not in the bundle:", JOBS["missing_assets"])
print("bundle ready:", sorted(os.listdir(f"{ROOT}/{BUNDLE}")))

In [ ]:
#@title 3. Detect the GPU and pick the tier
import subprocess
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise SystemExit("No GPU. Runtime > Change runtime type > pick a GPU (T4, L4, A100 or H100), then re-run.")
GPU_NAME, mib = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
VRAM_GB = round(float(mib) / 1024)
ORDER = ["draft", "standard", "max"]
fits = [t for t in ORDER if t in JOBS["tiers"] and JOBS["tiers"][t]["min_vram_gb"] <= VRAM_GB] or ["draft"]
CHOSEN = fits[-1] if TIER == "auto" else TIER
if ORDER.index(CHOSEN) > ORDER.index(JOBS["manifest_tier"]): CHOSEN = JOBS["manifest_tier"]
print(f"{GPU_NAME} {VRAM_GB} GB -> tier {CHOSEN} (manifest tier {JOBS['manifest_tier']})")
for n in JOBS["tiers"][CHOSEN]["notes"]: print("note:", n)

In [ ]:
#@title 4. Install ComfyUI and custom nodes at pinned commits
COMFY = f"{ROOT}/ComfyUI"
NODES = json.load(open(f"{ROOT}/{BUNDLE}/nodes.lock.json"))
def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True)
    if r.returncode: print("  (exit", r.returncode, ")")
    return r.returncode
if not os.path.isdir(COMFY):
    sh(f"git clone -q {NODES['comfyui']['repo']} {COMFY}")
sh(f"cd {COMFY} && git fetch -q origin && git checkout -q {NODES['comfyui']['commit']}")
sh(f"cd {COMFY} && pip -q install -r requirements.txt")
for url, sha in NODES["packs"].items():
    d = f"{COMFY}/custom_nodes/" + url.rstrip("/").split("/")[-1]
    if not os.path.isdir(d): sh(f"git clone -q {url} {d}")
    sh(f"cd {d} && git fetch -q origin && git checkout -q {sha}" if sha else f"echo unpinned {url}")
    if os.path.isfile(f"{d}/requirements.txt"): sh(f"pip -q install -r {d}/requirements.txt")
for d in NODES.get("bundled", []):  # node packs shipped inside the skills collection
    sh(f"rm -rf {COMFY}/custom_nodes/{d} && cp -r {ROOT}/{BUNDLE}/custom_nodes/{d} {COMFY}/custom_nodes/{d}")
    if os.path.isfile(f"{COMFY}/custom_nodes/{d}/requirements.txt"): sh(f"pip -q install -r {COMFY}/custom_nodes/{d}/requirements.txt")
print("installed", len(NODES["packs"]), "pinned packs and", len(NODES.get("bundled", [])), "bundled packs")

In [ ]:
#@title 5. Download model files (your own HF token from Google Colab secrets)
import urllib.request, shutil
def hf_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")
TOKEN = hf_token()
print("HF token:", "found (not shown)" if TOKEN else "none: gated repos will fail; add the HF_TOKEN secret")
MODELS = json.load(open(f"{ROOT}/{BUNDLE}/models.json"))["files"]
need = JOBS["tiers"][CHOSEN]["models"]; skipped = []
for f in need:
    e = MODELS.get(f) or {}
    dst = f"{COMFY}/models/{e.get('dir', 'diffusion_models').split('/')[0]}/{f}" if not e.get("dir", "").startswith("custom_nodes") else f"{COMFY}/{e['dir']}/{f.split('/')[-1]}"
    if os.path.isfile(dst): print("have", f); continue
    if not e.get("url"): skipped.append(f); continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(f"get  {f} ({(e.get('bytes') or 0)/1e9:.1f} GB) licence: {e.get('license')}")
    req = urllib.request.Request(e["url"], headers={"Authorization": f"Bearer {TOKEN}"} if TOKEN else {})
    try:
        with urllib.request.urlopen(req, timeout=120) as r, open(dst + ".part", "wb") as out: shutil.copyfileobj(r, out, 1 << 24)
        os.replace(dst + ".part", dst)
    except Exception as ex:
        print("  FAILED:", type(ex).__name__, getattr(ex, "code", "")); skipped.append(f)
del TOKEN
if skipped: print("NOT downloaded (no URL or failed; place them by hand or the jobs that need them fail):", skipped)

In [ ]:
#@title 6. Start ComfyUI (127.0.0.1:8188) and wait until it answers
import threading
log = open(f"{ROOT}/comfyui.log", "a")
COMFY_PROC = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"], cwd=COMFY, stdout=log, stderr=log)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); print("ComfyUI is up"); break
    except Exception: time.sleep(2)
else: raise SystemExit("ComfyUI did not start; see /content/comfyui.log")

In [ ]:
#@title 7. Watchdog: release the runtime at the cap, when idle, or on a teardown request
def _watchdog():
    idle_since = None
    while True:
        time.sleep(30)
        try:
            from google.colab import runtime
        except Exception:
            runtime = None
        why = None
        if (time.time() - SESSION_T0) / 60 >= MAX_SESSION_MINUTES: why = "session cap"
        if os.path.isfile(f"{COMFY}/input/gcr_teardown_request.txt"): why = "teardown request"
        try:
            q = json.load(urllib.request.urlopen("http://127.0.0.1:8188/queue", timeout=5))
            busy = q.get("queue_running") or q.get("queue_pending")
        except Exception:
            busy = False
        idle_since = None if busy else (idle_since or time.time())
        if idle_since and (time.time() - idle_since) / 60 >= IDLE_MINUTES and not RUNNER_ACTIVE: why = "idle"
        if why:
            print(f"watchdog: {why}; collecting outputs and releasing the runtime", flush=True)
            try: collect()
            except Exception as ex: print("collect failed:", ex)
            if runtime: runtime.unassign()
            return
RUNNER_ACTIVE = False
def collect():
    out = f"{ROOT}/{BUNDLE}/outputs"
    if not os.path.isdir(out): return None
    z = shutil.make_archive(f"{ROOT}/{BUNDLE}-outputs", "zip", out)
    if OUTPUT == "drive" and os.path.isdir("/content/drive/MyDrive"):
        shutil.copy2(z, "/content/drive/MyDrive/"); print("copied to Drive:", os.path.basename(z))
    return z
if OUTPUT == "drive":
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as ex: print("Drive not mounted:", ex)
threading.Thread(target=_watchdog, daemon=True).start()
print(f"watchdog on: cap {MAX_SESSION_MINUTES} min, idle {IDLE_MINUTES} min")

In [ ]:
#@title 8. (orchestrate mode) Open a tunnel so the director can reach ComfyUI
#@markdown Anyone with the tunnel URL can submit jobs to this ComfyUI while it is up. Share it only with your agent.
TUNNEL_URL = None
if MODE == "orchestrate" and TUNNEL == "cloudflared":
    if not os.path.isfile("/usr/local/bin/cloudflared"):
        sh("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared")
    import re
    tlog = open(f"{ROOT}/tunnel.log", "w")
    TUN = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"], stdout=tlog, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2); m_ = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{ROOT}/tunnel.log").read())
        if m_: TUNNEL_URL = m_.group(0); break
    print("Give this URL to the director:", TUNNEL_URL)
    print("  python skills/video-director/scripts/direct.py run <manifest> --target google-colab --url", TUNNEL_URL)
else:
    print("no tunnel (MODE is package, or TUNNEL is none)")

In [ ]:
#@title 9. (package mode) Render every job in the bundle (resumable)
if MODE == "package":
    RUNNER_ACTIVE = True
    try:
        rc = subprocess.run(["python", f"{ROOT}/{BUNDLE}/runner.py", "--tier", CHOSEN, "--out", f"{ROOT}/{BUNDLE}/outputs",
                             "--max-minutes", str(max(1, MAX_SESSION_MINUTES - (time.time() - SESSION_T0) / 60 - 3))]).returncode
    finally:
        RUNNER_ACTIVE = False
    print("runner exit", rc, "(0 = all done, 4 = some jobs failed or blocked, 5 = ComfyUI unreachable)")
else:
    print("orchestrate mode: the director submits the jobs; leave this notebook open")

In [ ]:
#@title 10. Collect outputs (Drive or download)
z = collect()
if z and OUTPUT == "zip":
    from google.colab import files; files.download(z)
print("outputs:", z, "- back on your machine: python skills/google-colab-runner/scripts/colab.py ingest <manifest> <outputs.zip>")

In [ ]:
#@title 11. Tear down: release the runtime now (stops compute-unit use)
from google.colab import runtime
runtime.unassign()